# Compare predicted result to actual 2026 EDC LV line up

1. Extract list of artist in 2026 line up

In [1]:
# import
from bs4 import BeautifulSoup
import requests
import pandas as pd
import os

# ─── SET THIS EACH YEAR ───────────────────────────────────────────────────────
current_year = 2026   # The EDC lineup year to compare against
# ─────────────────────────────────────────────────────────────────────────────

In [2]:
edc_url = f'https://lasvegas.electricdaisycarnival.com/lineup/{current_year}'

In [ ]:
artists_data = []
year = current_year
url = edc_url

try:
    page = requests.get(url, timeout=30)
    soup = BeautifulSoup(page.content, 'html.parser')
    # Extract artist names from data-artist-name attributes
    artist_tags = soup.select('[data-artist-name]')
    artist_names = [tag.get('data-artist-name') for tag in artist_tags]
except Exception as e:
    print(f"Scrape failed ({e}) - will keep the existing lineup CSV if one exists.")
    artist_names = []

# Keep the ORIGINAL spelling and capitalization exactly as the lineup page
# writes it (only surrounding whitespace is stripped). The normalized matching
# key is derived from it in the next cell - do NOT lowercase here, or the
# original formatting is lost for good.
for artist in artist_names:
    display_name = str(artist).strip()
    artists_data.append({'year': year,
                         'artist': display_name,
                         'artist_display': display_name})

print(f"\nTotal artist entries collected: {len(artists_data)}")

In [4]:
# Create DataFrame from collected data
df_edc_actual = pd.DataFrame(artists_data)

# Preview the data
print(f"\nSample data:")
df_edc_actual.head(10)


Sample data:


,year,artist
0,2026,$ami g
1,2026,1991
2,2026,2b happy
3,2026,2dy4
4,2026,999999999
5,2026,99jakes
6,2026,a.m.c
7,2026,mc phantom
8,2026,abana
9,2026,juliet mendoza


In [ ]:
# Normalize + Deduplicate the scraped lineup
# ---------------------------------------------------------
# The lineup page renders each artist block several times (responsive/layout
# variants), so every artist comes back a multiple of 3 times - and artists
# booked in 2 or 3 slots come back 6 or 9 times. Left alone this inflates the
# row count ~3.3x (1,437 rows for 438 real artists).
#
# Two columns from here on:
#   artist          -> normalized MATCHING key ('&' -> 'and', lowercase, collapsed spaces)
#   artist_display  -> ORIGINAL spelling/capitalization, used for every report
# Dedupe happens on the key, so 'Above & Beyond' and 'above and beyond' collapse
# into one artist - and the first spelling seen is the one kept for display.
import re

def norm_name(s):
    """Matching key only - never report this to a human."""
    s = str(s).lower().strip().replace('&', 'and')
    return re.sub(r'\s+', ' ', s)

def clean_display(s):
    """Report-ready name: keeps case, accents and punctuation exactly as scraped,
    only tidying whitespace runs left behind by the page markup."""
    return re.sub(r'\s+', ' ', str(s).strip())

if len(df_edc_actual) > 0:
    before = len(df_edc_actual)
    # Older CSVs (scraped before this cell existed) have no display column.
    if 'artist_display' not in df_edc_actual.columns:
        df_edc_actual['artist_display'] = df_edc_actual['artist']
    df_edc_actual['artist_display'] = df_edc_actual['artist_display'].map(clean_display)
    df_edc_actual['artist'] = df_edc_actual['artist_display'].map(norm_name)
    df_edc_actual = (df_edc_actual
                     .drop_duplicates(subset=['year', 'artist'], keep='first')
                     .sort_values('artist')
                     .reset_index(drop=True))
    after = len(df_edc_actual)
    print(f"Deduplicated lineup: {before} rows -> {after} unique artists "
          f"({before - after} duplicate rows removed)")
else:
    print("No scraped rows to deduplicate - skipping.")

df_edc_actual.head(10)

In [5]:
compare_dir = f'../compare_to_{current_year}'
os.makedirs(compare_dir, exist_ok=True)

filename = f'{compare_dir}/{current_year}_edc_lineup.csv'
if len(df_edc_actual) > 0:
    df_edc_actual.to_csv(filename, index=False)
    print(f"Saved {len(df_edc_actual)} artists to {filename}")
else:
    print(f"Scrape returned 0 artists - keeping the existing {filename}")

Saved 1437 artists to ../compare_to_2026/2026_edc_lineup.csv


# Extract artists that were predicted to play in 2026 from prediction csv

In [6]:
compare_dir = f'../compare_to_{current_year}'
input_path = f"../data/result_prediction/edc_{current_year}_prediction.csv"
output_path = f"{compare_dir}/{current_year}_edc_prediction.csv"

# Extract rows where the "result" column is 1 and save to a new CSV file, which means those artists are predicted to be in the lineup. This will allow us to compare the predicted lineup with the actual lineup we just scraped.
df = pd.read_csv(input_path)
filtered = df[df["result"] == 1]
filtered.to_csv(output_path, index=False)

# Print the number of rows written to the new CSV file
print(f"Wrote {len(filtered)} rows to {output_path}")

Wrote 430 rows to ../compare_to_2026/2026_edc_prediction.csv


In [ ]:
compare_dir = f'../compare_to_{current_year}'
pred_path = f"{compare_dir}/{current_year}_edc_prediction.csv"
act_path = f"{compare_dir}/{current_year}_edc_lineup.csv"
out_path = f"{compare_dir}/{current_year}_results.csv"

df_pred = pd.read_csv(pred_path)
df_act = pd.read_csv(act_path)

# MATCH on the normalized key, REPORT the original spelling.
# Both sources carry `artist` (normalized) + `artist_display` (original), so
# 'Above & Beyond' still matches 'above and beyond' but is printed the way the
# artist actually writes it.

def display_map(df):
    """{normalized key -> original spelling} for one source."""
    display_col = ('artist_display' if 'artist_display' in df.columns else 'artist')
    mapping = {}
    for raw, shown in zip(df['artist'].dropna(), df[display_col].fillna(df['artist'])):
        mapping.setdefault(norm_name(raw), str(shown).strip())  # first spelling wins
    return mapping

actual_disp = display_map(df_act)
pred_disp = display_map(df_pred)
actual_set, pred_set = set(actual_disp), set(pred_disp)
actual_keys, pred_keys = sorted(actual_set), sorted(pred_set)

# The two lists differ in length, so pad the shorter one to build a flat table.
n_rows = max(len(actual_keys), len(pred_keys))
def pad(values):
    return list(values) + [None] * (n_rows - len(values))

results_df = pd.DataFrame({
    'actual_lineup':       pad([actual_disp[k] for k in actual_keys]),
    'predicted_lineup':    pad([pred_disp[k] for k in pred_keys]),
    'actual_in_predicted': pad([k in pred_set for k in actual_keys]),
    'predicted_in_actual': pad([k in actual_set for k in pred_keys]),
})
# Padded cells are NaN - they represent "no artist here", not a match.
for col in ('actual_in_predicted', 'predicted_in_actual'):
    results_df[col] = results_df[col].fillna(False).astype(bool)

results_df.to_csv(out_path, index=False)
print(f"Results saved to {out_path} ({len(actual_keys)} actual vs {len(pred_keys)} predicted artists).")

results_df.head(10)

In [ ]:
# Every name printed below is the ORIGINAL spelling - matching already happened
# on the normalized key in the previous cell.

# Calculate totals for actual in predicted
actual_in_pred_true = results_df['actual_in_predicted'].sum()
total_actual = results_df['actual_lineup'].notna().sum()
actual_in_pred_false = total_actual - actual_in_pred_true

print("--- RECALL (How many ACTUAL artists were correctly predicted?) ---")
print(f"True (Predicted correctly): {actual_in_pred_true}")
print(f"False (Missed by model): {actual_in_pred_false}")
print(f"Recall: {(actual_in_pred_true / total_actual) * 100:.2f}%")
print(f"Out of all the {total_actual} actual artists that played, the model successfully guessed {actual_in_pred_true} of them\n")


# Calculate totals for predicted in actual
pred_in_act_true = results_df['predicted_in_actual'].sum()
total_pred = results_df['predicted_lineup'].notna().sum()
pred_in_act_false = total_pred - pred_in_act_true

print("--- PRECISION (How many PREDICTED artists actually showed up?) ---")
print(f"True (Correct prediction): {pred_in_act_true}")
print(f"False (Incorrect prediction): {pred_in_act_false}")
print(f"Precision: {(pred_in_act_true / total_pred) * 100:.2f}%")
print(f"Out of all the {total_pred} artists the model predicted, {pred_in_act_true} of them actually played at EDC {current_year}\n")

# Calculate F1 Score if appropriate
precision = pred_in_act_true / total_pred if total_pred > 0 else 0
recall = actual_in_pred_true / total_actual if total_actual > 0 else 0
if precision + recall > 0:
    f1 = 2 * (precision * recall) / (precision + recall)
else:
    f1 = 0
    
print("--- OVERALL F1 SCORE (Harmonic Mean of Precision & Recall) ---")
print(f"F1 Score: {f1 * 100:.2f}%")

# 1. Artists that actually played AND the model predicted them (True Positives)
correctly_predicted = results_df[results_df['actual_in_predicted'] == True]['actual_lineup'].dropna().tolist()

# 2. Artists that actually played BUT the model missed them (False Negatives)
missed_artists = results_df[results_df['actual_in_predicted'] == False]['actual_lineup'].dropna().tolist()

# 3. Artists the model predicted BUT they didn't actually play (False Positives)
incorrect_predictions = results_df[results_df['predicted_in_actual'] == False]['predicted_lineup'].dropna().tolist()

# Sort case-insensitively so 'ARMNHMR' and 'aaron k' interleave alphabetically
# instead of all-caps names being grouped ahead of the rest.
alpha = lambda names: sorted(names, key=str.lower)

print(f"=== CORRECTLY PREDICTED ARTISTS ({len(correctly_predicted)}) ===")
print(', '.join(alpha(correctly_predicted)))

print(f"\n=== MISSED BY MODEL ({len(missed_artists)}) ===")
print(', '.join(alpha(missed_artists)))

print(f"\n=== INCORRECT PREDICTIONS ({len(incorrect_predictions)}) ===")
print(', '.join(alpha(incorrect_predictions)))